In [1]:
# 細胞 1: 匯入必要的套件
import cv2
import mediapipe as mp
import pandas as pd
import os
import time

# 初始化 MediaPipe Pose 模組
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
pose = mp_pose.Pose(static_image_mode=False,  # 影片模式
                    model_complexity=2,       # 使用最高複雜度以獲得更好的 3D 估計
                    smooth_landmarks=True,
                    min_detection_confidence=0.5,
                    min_tracking_confidence=0.5)

In [2]:
# 細胞 2: 定義輸入影片路徑、輸出資料夾和 CSV 路徑
video_path = 'ballet01.mp4'  # 替換成您的影片檔案路徑
track_name = 'track_1'          # 輸出資料夾和 CSV 名稱
track_folder = track_name       # 資料夾名稱
output_csv = f'{track_name}.csv'  # CSV 檔案名稱

# 創建資料夾
os.makedirs(track_folder, exist_ok=True)

# 檢查影片是否存在
if not os.path.exists(video_path):
    print(f"影片檔案 {video_path} 不存在，請確認路徑。")
else:
    print(f"開始處理影片: {video_path}")

開始處理影片: ballet01.mp4


In [3]:
# 細胞 3: 處理影片，拆分成圖片（crop 到 bounding box），提取 3D 骨架座標並準備資料
# 開啟影片
cap = cv2.VideoCapture(video_path)

# 獲取影片資訊
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# 準備 CSV 資料列表
pose_data = []
frame_count = 0  # 用於計數有偵測到姿勢的畫面

start_time = time.time()

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    # 轉換 BGR 到 RGB (MediaPipe 需要 RGB)
    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # 處理姿勢偵測
    results = pose.process(image_rgb)
    
    if results.pose_landmarks and results.pose_world_landmarks:
        # 計算 bounding box (使用 pose_landmarks 的 normalized 座標)
        landmarks = results.pose_landmarks.landmark
        visible_landmarks = [lm for lm in landmarks if lm.visibility > 0.5]
        
        if visible_landmarks:
            min_x = max(0, min(lm.x for lm in visible_landmarks) - 0.05)  # padding 0.05
            max_x = min(1, max(lm.x for lm in visible_landmarks) + 0.05)
            min_y = max(0, min(lm.y for lm in visible_landmarks) - 0.05)
            max_y = min(1, max(lm.y for lm in visible_landmarks) + 0.05)
            
            # 轉換為像素
            left = int(min_x * frame_width)
            right = int(max_x * frame_width)
            top = int(min_y * frame_height)
            bottom = int(max_y * frame_height)
            
            # crop 圖片（正規化舞者在 bounding box 中）
            cropped_frame = frame[top:bottom, left:right]
            
            # 圖片命名：frame_000001.jpg 等
            frame_count += 1
            frame_name = f'frame_{frame_count:06d}.jpg'
            frame_path = os.path.join(track_folder, frame_name)
            
            # 儲存 crop 後的圖片
            cv2.imwrite(frame_path, cropped_frame)
            
            # 提取 3D world landmarks (x, y, z in meters, relative to hip)
            world_landmarks = results.pose_world_landmarks.landmark
            frame_data = {
                'frame': frame_name
            }
            
            for idx in range(33):  # 0 to 32
                lm = world_landmarks[idx]
                # 格式為 ('x', 'y', 'z') 字串
                coord_str = f"('{lm.x}', '{lm.y}', '{lm.z}')"
                frame_data[f'p{idx}'] = coord_str
            
            pose_data.append(frame_data)
    
    # 可選：顯示進度
    if frame_count % 100 == 0:
        print(f"已處理 {frame_count} 幀")

cap.release()
cv2.destroyAllWindows()

# 計算處理時間和 FPS
end_time = time.time()
processing_time = end_time - start_time
if frame_count > 0:
    processed_fps = frame_count / processing_time
else:
    processed_fps = 0
print(f"處理完成。總有效畫面數: {frame_count}, 處理時間: {processing_time:.2f} 秒, 平均 FPS: {processed_fps:.2f}")

已處理 100 幀
已處理 200 幀
已處理 300 幀
已處理 400 幀
已處理 500 幀
已處理 600 幀
已處理 700 幀
已處理 800 幀
已處理 900 幀
已處理 1000 幀
已處理 1100 幀
已處理 1200 幀
已處理 1300 幀
已處理 1400 幀
已處理 1500 幀
已處理 1600 幀
已處理 1700 幀
已處理 1800 幀
已處理 1900 幀
已處理 2000 幀
已處理 2100 幀
已處理 2200 幀
已處理 2300 幀
已處理 2400 幀
已處理 2400 幀
已處理 2500 幀
已處理 2600 幀
已處理 2700 幀
已處理 2800 幀
已處理 2900 幀
已處理 3000 幀
已處理 3100 幀
已處理 3200 幀
已處理 3300 幀
已處理 3400 幀
已處理 3500 幀
已處理 3600 幀
已處理 3700 幀
已處理 3800 幀
已處理 3900 幀
已處理 4000 幀
已處理 4100 幀
已處理 4200 幀
已處理 4300 幀
已處理 4400 幀
已處理 4500 幀
已處理 4600 幀
已處理 4700 幀
處理完成。總有效畫面數: 4724, 處理時間: 1143.75 秒, 平均 FPS: 4.13


In [4]:
# 細胞 4: 將資料寫入 CSV
if pose_data:
    # 定義欄位順序：frame, p0 to p32
    columns = ['frame'] + [f'p{i}' for i in range(33)]
    df = pd.DataFrame(pose_data, columns=columns)
    df.to_csv(output_csv, index=False)
    print(f"資料已輸出到 {output_csv}")
    print(f"圖片已儲存到資料夾 {track_folder}")
else:
    print("沒有偵測到任何姿勢資料。")

資料已輸出到 track_1.csv
圖片已儲存到資料夾 track_1
